# ⚡ اجرای Google Chrome با شتاب WebGPU روی Google Colab (Tesla T4)

**فقط یکی از دو سلول زیر را Play کنید** (نه هر دو):

| سلول | روش | نیاز به اکانت | سرعت |
| :--- | :--- | :---: | :--- |
| **سلول ۱** | KasmVNC (بازکردن در مرورگر) | ❌ ندارد | خوب |
| **سلول ۲** | Chrome Remote Desktop | ✅ گوگل | خیلی روان |

> ⚠️ قبل از اجرا: از منوی بالا `Runtime` ➔ `Change runtime type` ➔ **T4 GPU** ➔ `Save`.

پیش‌فرض `TARGET_URL` روی `https://webgpureport.org` است که شتاب WebGPU را نشان می‌دهد؛ می‌توانید به هر آدرسی عوضش کنید.


In [ ]:
#@title 🟢 [روش اول - سریع] Selkies (WebSocket + H.264/NVENC) + Google Chrome WebGPU
#@markdown **بدون اکانت، بدون رمز.** فقط Play بزنید و منتظر لینک بمانید (~۱ تا ۳ دقیقه).
#@markdown این روش از هسته‌ی مدرن Selkies (ساخته‌شده توسط مهندسان گوگل) استفاده می‌کند: استریم روی WebSocket،
#@markdown انکود سخت‌افزاری H.264 با NVENC روی تسلا T4، و همگام‌سازی خودکار رزولوشن با پنجره‌ی شما.
TARGET_URL = "https://webgpureport.org" #@param {type:"string"}
RESOLUTION = "1280x720" #@param ["1280x720", "1366x768", "1600x900", "1920x1080"]

import os, re, time, json, ssl, socket, shutil, subprocess, urllib.request
from IPython.display import display, HTML

HOME = os.path.expanduser("~")
SELKIES_PORT = 8899            # 8080 روی کولب اشغال است
CF_LOG = "/tmp/cf_selkies.log"

def run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

def port_open(p):
    try:
        s = socket.create_connection(("127.0.0.1", p), timeout=1); s.close(); return True
    except OSError:
        return False

print("GPU:", run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader") or "none")

# ── 1. پکیج‌های سیستم + Chrome + درایور Vulkan ───────────────────────────
print("\n▶ [1/5] نصب پکیج‌های سیستم، Google Chrome و درایور Vulkan ...", flush=True)
subprocess.run("apt-get update -qq", shell=True)
subprocess.run("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq xvfb xfce4 xfce4-terminal "
               "pulseaudio xauth curl wget libgbm1 libnss3 libasound2t64 libvulkan1 >/dev/null 2>&1 || true", shell=True)
if not os.path.exists("/usr/bin/google-chrome"):
    subprocess.run("wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb && "
                   "(dpkg -i /tmp/chrome.deb >/dev/null 2>&1 || apt-get install -fy -qq >/dev/null 2>&1); rm -f /tmp/chrome.deb", shell=True)
print("   Chrome:", run("/usr/bin/google-chrome --version 2>&1") or "نصب نشد!")

os.makedirs("/etc/vulkan/icd.d", exist_ok=True)
open("/etc/vulkan/icd.d/nvidia_icd.json", "w").write(json.dumps({
    "file_format_version": "1.0.0",
    "ICD": {"library_path": "libGLX_nvidia.so.0", "api_version": "1.3.275"}
}, indent=2))
open("/etc/ld.so.conf.d/nvidia.conf", "w").write("/usr/lib64-nvidia\n")
subprocess.run("ldconfig", shell=True)

# ── 2. Selkies ───────────────────────────────────────────────────────────
# ── 1b. تم مدرن (Arc-Dark + آیکون Papirus) + والپیپر ─────────────────────
print("▶ [1b] اعمال تم مدرن دسکتاپ ...", flush=True)
subprocess.run("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq arc-theme papirus-icon-theme fonts-ubuntu >/dev/null 2>&1 || true", shell=True)
try:
    from PIL import Image, ImageDraw
    _img = Image.new("RGB", (1920, 1080)); _d = ImageDraw.Draw(_img)
    for _y in range(1080):
        _t = _y / 1080.0
        _d.line([(0, _y), (1920, _y)], fill=(int(18 + 30 * _t), int(24 + 34 * _t), int(38 + 60 * _t)))
    os.makedirs("/usr/share/backgrounds", exist_ok=True)
    _img.save("/usr/share/backgrounds/selkies-modern.png")
except Exception:
    pass
_xf = HOME + "/.config/xfce4/xfconf/xfce-perchannel-xml"
os.makedirs(_xf, exist_ok=True)
open(_xf + "/xsettings.xml", "w").write(
    '<?xml version="1.0"?><channel name="xsettings" version="1.0"><property name="Net" type="empty">'
    '<property name="ThemeName" type="string" value="Arc-Dark"/>'
    '<property name="IconThemeName" type="string" value="Papirus-Dark"/></property></channel>')
open(_xf + "/xfwm4.xml", "w").write(
    '<?xml version="1.0"?><channel name="xfwm4" version="1.0"><property name="general" type="empty">'
    '<property name="theme" type="string" value="Arc-Dark"/></property></channel>')
open(_xf + "/xfce4-desktop.xml", "w").write(
    '<?xml version="1.0"?><channel name="xfce4-desktop" version="1.0"><property name="backdrop" type="empty">'
    '<property name="screen0" type="empty"><property name="monitor0" type="empty"><property name="workspace0" type="empty">'
    '<property name="last-image" type="string" value="/usr/share/backgrounds/selkies-modern.png"/>'
    '<property name="image-style" type="int" value="5"/></property></property></property></property></channel>')

print("▶ [2/5] نصب Selkies ...", flush=True)
if not shutil.which("selkies-session"):
    subprocess.run("pip install -q selkies", shell=True)
print("   selkies-session:", shutil.which("selkies-session") or "MISSING")

# ── 3. اجرای خودکار Chrome با شتاب WebGPU در دسکتاپ Selkies ──────────────
print("▶ [3/5] تنظیم اجرای خودکار Chrome با فلگ‌های WebGPU ...", flush=True)
os.makedirs(HOME + "/.config/autostart", exist_ok=True)
os.makedirs(HOME + "/.config/google-chrome", exist_ok=True)
open(HOME + "/.config/google-chrome/First Run", "w").close()
open(HOME + "/launch_chrome.sh", "w").write(
    "#!/bin/bash\n"
    "export VK_ICD_FILENAMES=/etc/vulkan/icd.d/nvidia_icd.json\n"
    "google-chrome-stable --no-sandbox --no-first-run --no-default-browser-check --password-store=basic "
    "--disable-gpu-sandbox --enable-unsafe-webgpu --enable-features=Vulkan,DefaultANGLEVulkan,VulkanFromANGLE,WebGPUService "
    "--use-angle=vulkan --enable-dawn-features=allow_unsafe_apis,disable_adapter_blocklist --ignore-gpu-blocklist "
    "--disable-dev-shm-usage --enable-gpu-rasterization --enable-zero-copy --start-maximized \"" + TARGET_URL + "\" &\n")
os.chmod(HOME + "/launch_chrome.sh", 0o755)
open(HOME + "/.config/autostart/webgpu-chrome.desktop", "w").write(
    "[Desktop Entry]\nType=Application\nName=WebGPU Google Chrome\nExec=" + HOME + "/launch_chrome.sh\nTerminal=false\n")
subprocess.run("rm -f /etc/xdg/autostart/gnome-keyring-*.desktop /usr/share/dbus-1/services/*keyring* "
               "/usr/share/dbus-1/services/org.freedesktop.secrets.service 2>/dev/null; true", shell=True)

# ── 4. راه‌اندازی Selkies ─────────────────────────────────────────────────
print("▶ [4/5] راه‌اندازی Selkies (بدون رمز عبور، انکود سخت‌افزاری H.264) ...", flush=True)
subprocess.run("pkill -9 -f selkies >/dev/null 2>&1; sleep 2; true", shell=True)
# WebRTC (روانترین) + انکود NVENC + کیفیت بالا (۴:۴:۴ برای متن شارپ، بیت‌ریت و CRF بهتر)
SELKIES_FLAGS = ("--session=xfce --public --port=" + str(SELKIES_PORT) +
                 " --enable-basic-auth=false --mode=webrtc --encoder=h264enc "
                 " --video-fullcolor=true --video-bitrate=20000 --video-crf=18 "
                 " --framerate=60 --enable-resize=true")
subprocess.Popen("setsid nohup selkies-session " + SELKIES_FLAGS + " > /tmp/selkies.log 2>&1 &", shell=True)
for _ in range(30):
    if port_open(SELKIES_PORT): break
    time.sleep(2)
if not port_open(SELKIES_PORT):
    print("❌ خطای Selkies:")
    print(run("tail -n 25 /tmp/selkies.log"))
    raise RuntimeError("Selkies failed to start")
print("   ✔ Selkies روی پورت", SELKIES_PORT, "فعال شد")

# ── 5. تونل Cloudflare (با استفاده‌ی مجدد تا لینک ثابت بماند) ─────────────
print("▶ [5/5] ساخت/اتصال تونل Cloudflare ...", flush=True)
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run("wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared", shell=True)

def _alive(u):
    try:
        urllib.request.urlopen(u, timeout=10, context=ssl._create_unverified_context()); return True
    except Exception as e:
        return getattr(e, "code", None) in (200, 401, 403)

existing = None
if subprocess.run("pgrep -f cloudflared", shell=True, capture_output=True).returncode == 0 and os.path.exists(CF_LOG):
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open(CF_LOG).read())
    if m: existing = m.group(0)

tunnel_url = None
if existing and _alive(existing):
    tunnel_url = existing
    print("   ✔ استفاده مجدد از تونل فعال (لینک ثابت):", tunnel_url)
else:
    subprocess.run("pkill -9 -f cloudflared >/dev/null 2>&1; sleep 1; true", shell=True)
    if os.path.exists(CF_LOG): os.remove(CF_LOG)
    subprocess.Popen("setsid nohup /usr/local/bin/cloudflared tunnel --no-autoupdate --url http://127.0.0.1:" +
                     str(SELKIES_PORT) + " --logfile " + CF_LOG + " >/dev/null 2>&1 &", shell=True)
    for _ in range(35):
        time.sleep(1)
        if os.path.exists(CF_LOG):
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open(CF_LOG).read())
            if m: tunnel_url = m.group(0); break

print("\n" + "═" * 65)
if tunnel_url:
    print("🎉 دسکتاپ سریع Selkies آماده است! (بدون نیاز به ورود)")
    print("🔗", tunnel_url)
    print("═" * 65)
    display(HTML(
        '<div style="background:linear-gradient(135deg,#0f2027,#203a43,#2c5364);padding:22px;border-radius:12px;'
        'color:#fff;font-family:sans-serif;text-align:center">'
        '<h2 style="margin:0 0 10px 0;color:#00e676">⚡ دسکتاپ سریع Selkies آماده است</h2>'
        '<p style="opacity:.9">بدون رمز — فقط روی دکمه بزنید. برای کنسول تنظیمات، منوی کنار صفحه را باز کنید.</p>'
        '<a href="' + tunnel_url + '" target="_blank" style="padding:14px 34px;background:#00e676;color:#000;'
        'font-weight:bold;font-size:18px;text-decoration:none;border-radius:8px;display:inline-block">'
        '👉 ورود به دسکتاپ (Selkies)</a>'
        '<p style="margin-top:12px;font-size:13px;opacity:.8"><a href="' + tunnel_url + '" target="_blank" '
        'style="color:#64ffda">' + tunnel_url + '</a></p></div>'))
else:
    print("❌ خطا در ساخت تونل Cloudflare")
    print(run("tail -n 20 " + CF_LOG))


In [ ]:
#@title ⚡ [روش دوم] راه‌اندازی با Google Chrome Remote Desktop (Live Real-Time Logs)
#@markdown ### 📋 راهنمای اتصال ریموت دسکتاپ گوگل (تاخیر زیر ۳۰ میلی‌ثانیه و ۶۰ فریم):
#@markdown 1. در یک تب جدید در کامپیوتر خود به **[remotedesktop.google.com/headless](https://remotedesktop.google.com/headless)** بروید.
#@markdown 2. دکمه‌های **Begin** ➔ **Next** ➔ **Authorize** را بزنید و کد لینوکس دبیان را کپی کنید.
#@markdown ⚠️ توجه: کد یک‌بار مصرف است؛ دقیقاً قبل از اجرای سلول آن را دریافت و پیست کنید.
AUTH_COMMAND = "" #@param {type:"string"}
PIN = 123456 #@param {type:"integer"}
TARGET_URL = "https://webgpureport.org" #@param {type:"string"}

import os, re, time, subprocess
from IPython.display import display, HTML

# پین باید همیشه ۶ رقمی رشته‌ای باشد
PIN = str(PIN).zfill(6)[:6]

raw_auth = AUTH_COMMAND.strip()
if not raw_auth:
    print("⚠️ فیلد AUTH_COMMAND خالی است!")
    print("👈 دریافت کد لینوکس: https://remotedesktop.google.com/headless")
    try:
        raw_auth = input("دستور احراز هویت را وارد کنید: ").strip()
    except (EOFError, OSError):
        raw_auth = ""

if not raw_auth:
    raise ValueError("دستور احراز هویت الزامی است.")

def run_step(step_num, title, cmd):
    print(f"\n▶ [{step_num}/5] {title}...", flush=True)
    t0 = time.time()
    code = subprocess.run(cmd, shell=True).returncode
    dt = time.time() - t0
    status = "✔ انجام شد" if code == 0 else f"✖ خطا (کد {code})"
    print(f"  └─ {status} در {dt:.1f} ثانیه", flush=True)
    return code

# ── 1. درایورهای گرافیک و Vulkan ─────────────────────────────────────────
run_step(1, "پیکربندی سخت‌افزاری هسته لینوکس، لودر Vulkan و درایور تسلا T4", """
mkdir -p /dev/dri /etc/vulkan/icd.d /usr/share/vulkan/icd.d
mknod -m 666 /dev/dri/card0 c 226 0 2>/dev/null || true
mknod -m 666 /dev/dri/renderD128 c 226 128 2>/dev/null || true
mknod -m 666 /dev/nvidia-modeset c 195 254 2>/dev/null || true
chmod -R 666 /dev/dri /dev/nvidia* 2>/dev/null || true

apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libvulkan1 vulkan-tools dbus-x11 >/dev/null 2>&1

cat << 'EOF' > /etc/vulkan/icd.d/nvidia_icd.json
{
  "file_format_version": "1.0.0",
  "ICD": {
    "library_path": "libGLX_nvidia.so.0",
    "api_version": "1.3.275"
  }
}
EOF
cp /etc/vulkan/icd.d/nvidia_icd.json /usr/share/vulkan/icd.d/nvidia_icd.json
echo '/usr/lib64-nvidia' > /etc/ld.so.conf.d/nvidia.conf
ldconfig
/etc/init.d/dbus start >/dev/null 2>&1 || true
""")

# ── 2. کاربر اختصاصی ─────────────────────────────────────────────────────
run_step(2, "ساخت کاربر اختصاصی colab و دسترسی‌های سیستمی", """
mkdir -p /home/colab
id -u colab >/dev/null 2>&1 || useradd -m -s /bin/bash colab
echo 'colab:123456' | chpasswd
usermod -aG sudo,video,render colab 2>/dev/null || true
echo 'colab ALL=(ALL) NOPASSWD:ALL' > /etc/sudoers.d/colab
""")

# ── 3. پکیج‌های XFCE و Chrome Remote Desktop ──────────────────────────────
run_step(3, "نصب پکیج‌های دسکتاپ XFCE4 و Chrome Remote Desktop", """
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq xfce4 desktop-base xfce4-terminal >/dev/null 2>&1
DEBIAN_FRONTEND=noninteractive apt-get purge -y -qq xscreensaver >/dev/null 2>&1 || true
# کتابخانه‌های گرافیکی که باینری start-host لازم دارد (روی Ubuntu 24.04 پسوند t64)
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libatk1.0-0t64 libatk-bridge2.0-0t64 libgtk-3-0t64 libcups2t64 libxcomposite1 libxdamage1 libxrandr2 libxkbcommon0 libpango-1.0-0 libcairo2 libgbm1 libnss3 libasound2t64 libxfixes3 >/dev/null 2>&1 || \
  DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libatk1.0-0 libatk-bridge2.0-0 libgtk-3-0 libcups2 libxcomposite1 libxdamage1 libxrandr2 libxkbcommon0 libpango-1.0-0 libcairo2 libgbm1 libnss3 libasound2 libxfixes3 >/dev/null 2>&1 || true
# اگر start-host هنوز کتابخانه کم دارد، همین‌جا هشدار واضح بده
if ! ldd /opt/google/chrome-remote-desktop/start-host >/dev/null 2>&1; then
  echo "⚠️ start-host still missing libraries:"; ldd /opt/google/chrome-remote-desktop/start-host 2>&1 | grep "not found" || true
fi
if [ ! -f /opt/google/chrome-remote-desktop/start-host ]; then
  wget -q -O /tmp/crd.deb https://dl.google.com/linux/direct/chrome-remote-desktop_current_amd64.deb
  dpkg -i /tmp/crd.deb >/dev/null 2>&1 || apt-get install -fy -qq >/dev/null 2>&1
  rm -f /tmp/crd.deb
fi
usermod -aG chrome-remote-desktop colab 2>/dev/null || true

cat << 'EOF' > /home/colab/.chrome-remote-desktop-session
export DESKTOP_SESSION=xfce
export XDG_CURRENT_DESKTOP=XFCE
export XDG_CONFIG_DIRS=/etc/xdg/xdg-xfce:/etc/xdg
exec /usr/bin/xfce4-session
EOF
cp /home/colab/.chrome-remote-desktop-session /etc/chrome-remote-desktop-session
chmod +x /home/colab/.chrome-remote-desktop-session
""")

# ── 4. کروم و اسکریپت WebGPU در Startup دسکتاپ ───────────────────────────
run_step(4, "نصب Google Chrome و تنظیم اجرای خودکار با شتاب Dawn WebGPU", f"""
if [ ! -f /usr/bin/google-chrome ]; then
  wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
  dpkg -i /tmp/chrome.deb >/dev/null 2>&1 || apt-get install -fy -qq >/dev/null 2>&1
  rm -f /tmp/chrome.deb
fi
rm -f /etc/xdg/autostart/gnome-keyring-*.desktop
rm -f /usr/share/dbus-1/services/*keyring* /usr/share/dbus-1/services/org.freedesktop.secrets.service 2>/dev/null || true
mkdir -p /home/colab/.config/autostart /home/colab/Desktop /home/colab/.config/google-chrome
touch "/home/colab/.config/google-chrome/First Run"
cat << 'EOF' > /home/colab/launch_chrome.sh
#!/bin/bash
export VK_ICD_FILENAMES=/etc/vulkan/icd.d/nvidia_icd.json
google-chrome-stable \\
  --no-first-run \\
  --no-default-browser-check \\
  --password-store=basic \\
  --no-sandbox \\
  --disable-gpu-sandbox \\
  --enable-unsafe-webgpu \\
  --enable-features=Vulkan,DefaultANGLEVulkan,VulkanFromANGLE,WebGPUService \\
  --use-angle=vulkan \\
  --enable-dawn-features=allow_unsafe_apis,disable_adapter_blocklist \\
  --disable-dawn-features=disallow_unsafe_apis \\
  --ignore-gpu-blocklist \\
  --disable-dev-shm-usage \\
  --enable-gpu-rasterization \\
  --enable-zero-copy \\
  --start-maximized \\
  "{TARGET_URL}" &
EOF
chmod +x /home/colab/launch_chrome.sh
cat << 'EOF' > /home/colab/.config/autostart/webgpu-chrome.desktop
[Desktop Entry]
Version=1.0
Type=Application
Name=WebGPU Google Chrome
Exec=/home/colab/launch_chrome.sh
Icon=google-chrome
Terminal=false
EOF
cp /home/colab/.config/autostart/webgpu-chrome.desktop /home/colab/Desktop/
chmod +x /home/colab/Desktop/webgpu-chrome.desktop
chown -R colab:colab /home/colab
""")

# ── 5. استارت هاست با ثبت پین از طریق stdin ──────────────────────────────
print("\n▶ [5/5] ثبت و راه‌اندازی هاست ریموت دسکتاپ گوگل با اکانت شما...", flush=True)

# پاک‌سازی دستور گوگل:
#  - پیشوند DISPLAY= حذف می‌شود (در نسخه‌های جدید CRD باعث unrecognized argument می‌شد)
#  - --pin=... حذف؛ پین از طریق stdin تغذیه می‌شود
clean_cmd = re.sub(r'^DISPLAY=\s*', '', raw_auth)
clean_cmd = re.sub(r'--pin=\S+', '', clean_cmd).strip()

runner_script = f"""#!/bin/bash
printf '{PIN}\\n{PIN}\\n' | {clean_cmd}
sleep 2
/opt/google/chrome-remote-desktop/chrome-remote-desktop --start || true
"""
with open("/tmp/start_crd.sh", "w", encoding="utf-8") as f:
    f.write(runner_script)
subprocess.run("chmod 755 /tmp/start_crd.sh && chown colab:colab /tmp/start_crd.sh", shell=True)

res = subprocess.run("su - colab -s /bin/bash -c /tmp/start_crd.sh",
                     shell=True, capture_output=True, text=True)
time.sleep(3)
check = subprocess.run("ps aux | grep -v grep | grep chrome-remote-desktop",
                       shell=True, capture_output=True, text=True)

print("\n" + "═" * 65)
if "chrome-remote-desktop" in check.stdout:
    print("🎉 سرویس Google Chrome Remote Desktop با موفقیت استارت شد!")
    print(f"🔑 پین اتصال: {PIN}")
    print("═" * 65)
    banner = f"""
        <div style="background: linear-gradient(135deg, #0d47a1 0%, #1976d2 100%); padding: 24px; border-radius: 12px; color: #fff; font-family: sans-serif; box-shadow: 0 4px 20px rgba(0,0,0,0.3); text-align: center; margin: 15px 0;">
            <h2 style="margin: 0 0 10px 0; color: #64ffda;">🚀 دسکتاپ ابری ۶۰ فریم آماده اتصال است!</h2>
            <p style="font-size: 15px; margin-bottom: 20px; opacity: 0.95;">
                دستگاه در پنل ریموت گوگل ثبت شد. روی دکمه زیر کلیک کرده و پین <b>{PIN}</b> را وارد کنید:
            </p>
            <a href="https://remotedesktop.google.com/access" target="_blank" style="padding: 14px 34px; background: #00e676; color: #000; font-weight: bold; font-size: 18px; text-decoration: none; border-radius: 8px; display: inline-block; box-shadow: 0 4px 15px rgba(0,230,118,0.4);">
                👉 ورود به Chrome Remote Desktop
            </a>
            <p style="margin-top: 15px; font-size: 13px; opacity: 0.8;">
                آدرس مستقیم: <a href="https://remotedesktop.google.com/access" target="_blank" style="color: #64ffda;">https://remotedesktop.google.com/access</a>
            </p>
        </div>
    """
    display(HTML(banner))
else:
    print("⚠️ خروجی فراخوانی:")
    print(res.stdout or "(خالی)")
    if res.stderr: print("خطاها:", res.stderr)
    print("═" * 65)
    print("نکته مهم: کد تأیید گوگل یک‌بار مصرف است. مطمئن شوید دقیقاً قبل از اجرای سلول آن را از remotedesktop.google.com/headless کپی کرده‌اید.")
